# FinGuard — IBM LinuxONE Datathon

**Run the three code cells from top to bottom using Shift + Enter. No terminal is needed.**

This notebook checks the actual files on your LinuxONE server, checks for evidence of previous training, trains and evaluates a fraud model, and creates a standalone interactive website.

**Important:** Your original CSVs are not modified. All results are calculated from the real files available to your Jupyter kernel.

## Step 1 — Check your files and model status

Run the next code cell. It displays filenames, the installed packages, and any saved model/notebook training evidence, without displaying private transaction records.

In [ ]:
from pathlib import Path
from importlib.util import find_spec
from finguard_pipeline import locate_data, check_training_state

print('Jupyter working folder:', Path.cwd())
for filename, location in locate_data('/home/ibm-user').items():
    print(f'  {filename}: {location if location else "NOT FOUND"}')

for package in ('pandas', 'numpy', 'sklearn', 'joblib'):
    print(f'  {package}: {"available" if find_spec(package) else "MISSING"}')

check = check_training_state('/home/ibm-user')
print('\nSaved model/evaluation files:', [item['name'] for item in check['saved_artifacts']])
print('Saved notebook training evidence:')
for item in check['notebook_checks']:
    print(' ', item)
print('\nNote: A cell might have executed without saving a model. FinGuard will train a fresh, reproducible baseline.')

## Step 2 — Train and test the model; build the website

This runs against `transactions_history.csv`, not made-up data. We split the usable labeled records into training, validation and test sets, select the threshold on validation, then report held-out test performance. Training may take a minute on a busy server.

If you get `ModuleNotFoundError`, ask an IBM mentor whether `pandas`, `numpy`, `scikit-learn` and `joblib` can be installed in this Jupyter kernel.

In [ ]:
from pathlib import Path
from finguard_pipeline import run_pipeline

result = run_pipeline(data_dir='/home/ibm-user', output_dir=Path.cwd())

print('\n=== SOURCE DATA SUMMARY ===')
print('Transactions:', format(result['summary']['rows'], ','))
print('Labeled fraud:', format(result['summary']['fraud_count'] or 0, ','))
print('Source fraud rate:', result['summary']['fraud_rate'], '%')
if result['metrics']:
    print('\n=== ACTUAL HELD-OUT MODEL RESULTS ===')
    for key in ('rows','positives','pr_auc','precision','recall','f1','roc_auc','tp','fp','tn','fn','threshold'):
        print(f'{key}: {result["metrics"][key]}')
else:
    print('The source labels were missing or insufficient for training. Read the message above and the methodology panel.')
print('\nWebsite generated:', result['report'])

## Step 3 — Open the interactive FinGuard website

Run the next cell. It gives you a link to the HTML website and tries to display it inline. **If the embedded preview is blank, click the link instead.** Your data never leaves the LinuxONE workspace unless you download the generated report.

In [ ]:
from pathlib import Path
from IPython.display import display, HTML, IFrame, FileLink

report = Path(result['report'])
print('Website is ready:', report)
display(HTML('<h3>FinGuard website</h3><p>Open the link below for your real-data interactive report.</p>'))
display(FileLink(report.name))
display(IFrame(src=report.name, width='100%', height=740))

### Generated files

- `finguard_report.html` — interactive website with graphs, holdout case review, filters, threshold slider and CSV export
- `finguard_metrics.json` — verified test results for your pitch
- `finguard_model.joblib` — fitted model, generated only when training works

Your CSV source files are unchanged. The generated report contains selected transaction amounts and case scores, **not raw client IDs or card IDs**. Keep it within the approved datathon environment.

If you only see a JSON text editor and cannot execute notebook cells, you need a working Jupyter Python kernel. Ask an IBM mentor to open the notebook in Notebook view, not Text Editor.